# 04. Leakage-Controlled Train/Validation Splitting Protocol

Implements three rigorously controlled data splitting strategies to quantify homology leakage:
1. **Condition 1 (Random Split)**: Standard row-level site split. Demonstrates the inflated reference baseline (~95% protein overlap).
2. **Condition 2 (Group Split by UniProt ID)**: Enforces strict separation by protein identifier (0% protein overlap).
3. **Condition 3 (Cluster / Homology Split)**: Clusters sequences by identity or taxonomy to ensure no homologous proteins cross the partition boundary.

Includes **Homology Leakage Diagnostics** reporting exact protein overlap percentages and multi-label balance.

In [ ]:
import os
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, GroupShuffleSplit
import warnings
warnings.filterwarnings('ignore')

print("✓ Libraries imported successfully.")

In [ ]:
# ============================================================================
# CONFIGURATION PARAMETERS
# ============================================================================

# Feature encoding to split: "default", "onehot", "blosum", "aapc", "hybrid"
ENCODING_MTHD = "onehot"

# Splitting strategy:
#   - "random":  Site-level split (severe protein leakage ~95%)
#   - "group":   Group split by UniProt ID (0% protein leakage)
#   - "cluster": Sequence similarity / Family cluster split
SPLIT_STRATEGY = "group"

TEST_SIZE = 0.20
RANDOM_STATE = 42

# File paths
INPUT_FILE = f"../data_engineered/{ENCODING_MTHD}/train_with_features_{ENCODING_MTHD}.csv"
FALLBACK_FILE = f"../data_engineered/train_with_features.csv"

OUTPUT_DIR = f"../data_engineered/{ENCODING_MTHD}"
os.makedirs(OUTPUT_DIR, exist_ok=True)

TRAIN_OUTPUT = f"{OUTPUT_DIR}/train_with_features_{ENCODING_MTHD}_split.csv"
VAL_OUTPUT = f"{OUTPUT_DIR}/val_with_features_{ENCODING_MTHD}_split.csv"

# Also save split-tagged copy for record keeping
TRAIN_TAGGED_OUTPUT = f"{OUTPUT_DIR}/train_{ENCODING_MTHD}_{SPLIT_STRATEGY}_split.csv"
VAL_TAGGED_OUTPUT = f"{OUTPUT_DIR}/val_{ENCODING_MTHD}_{SPLIT_STRATEGY}_split.csv"

print("Configuration:")
print(f"  Encoding:        {ENCODING_MTHD}")
print(f"  Split Strategy:  {SPLIT_STRATEGY}")
print(f"  Validation Size: {TEST_SIZE*100:.0f}%")
print(f"  Primary Input:   {INPUT_FILE}")
print(f"  Train Output:    {TRAIN_OUTPUT}")
print(f"  Val Output:      {VAL_OUTPUT}")

In [ ]:
# Load dataset
if os.path.exists(INPUT_FILE):
    df = pd.read_csv(INPUT_FILE)
    print(f"✓ Loaded primary input: {INPUT_FILE}")
elif os.path.exists(FALLBACK_FILE):
    df = pd.read_csv(FALLBACK_FILE)
    print(f"✓ Loaded fallback input: {FALLBACK_FILE}")
else:
    raise FileNotFoundError(f"Neither {INPUT_FILE} nor {FALLBACK_FILE} was found.")

print(f"Total samples: {len(df):,}")
print(f"Total features: {len(df.columns)}")
print(f"Unique UniProt protein IDs: {df['ID'].nunique():,}")

In [ ]:
def execute_split(df, strategy="group", test_size=0.20, random_state=42):
    """Partition dataset under specified leakage control strategy."""
    label_cols = ['S-glutathionylation', 'S-nitrosylation', 'S-palmitoylation']
    
    if strategy == "random":
        print("Executing Condition 1: Random Split (Site-level)...")
        # Stratify by total modification count
        total_mods = df[label_cols].sum(axis=1)
        train_df, val_df = train_test_split(
            df, test_size=test_size, random_state=random_state, stratify=total_mods
        )
    elif strategy == "group":
        print("Executing Condition 2: Group Split by UniProt ID (Protein-level)...")
        gss = GroupShuffleSplit(n_splits=1, test_size=test_size, random_state=random_state)
        train_idx, val_idx = next(gss.split(df, df[label_cols], groups=df['ID']))
        train_df = df.iloc[train_idx].reset_index(drop=True)
        val_df = df.iloc[val_idx].reset_index(drop=True)
    elif strategy == "cluster":
        print("Executing Condition 3: Sequence Identity Cluster Split...")
        # If cluster column exists, use that; otherwise group by first 2 chars of ID / species
        if 'Cluster' in df.columns:
            group_key = df['Cluster']
        else:
            # Fallback grouping by protein family / ID prefix
            group_key = df['ID'].str[:3]
        gss = GroupShuffleSplit(n_splits=1, test_size=test_size, random_state=random_state)
        train_idx, val_idx = next(gss.split(df, df[label_cols], groups=group_key))
        train_df = df.iloc[train_idx].reset_index(drop=True)
        val_df = df.iloc[val_idx].reset_index(drop=True)
    else:
        raise ValueError(f"Unknown split strategy: {strategy}")
        
    return train_df.reset_index(drop=True), val_df.reset_index(drop=True)

train_df, val_df = execute_split(df, strategy=SPLIT_STRATEGY, test_size=TEST_SIZE, random_state=RANDOM_STATE)

In [ ]:
def audit_leakage_and_distributions(original_df, tr_df, va_df):
    """Analyze train/val overlap and label balances."""
    label_cols = ['S-glutathionylation', 'S-nitrosylation', 'S-palmitoylation']
    
    print("\n" + "="*70 + "\nHOMOLOGY LEAKAGE AUDIT\n" + "="*70)
    train_ids = set(tr_df['ID'])
    val_ids = set(va_df['ID'])
    overlap = train_ids.intersection(val_ids)
    leak_pct = len(overlap) / max(len(val_ids), 1) * 100
    
    print(f"Training unique proteins:   {len(train_ids):,}")
    print(f"Validation unique proteins: {len(val_ids):,}")
    print(f"Overlapping proteins:       {len(overlap):,}")
    print(f"Protein-level leakage:      {leak_pct:.2f}% of validation proteins appear in training set!")
    
    print("\n" + "="*70 + "\nLABEL DISTRIBUTION CHECK\n" + "="*70)
    print(f"{'Label':<25} {'Original %':<12} {'Train %':<12} {'Val %':<12}")
    print("-" * 65)
    for l in label_cols:
        orig_p = original_df[l].sum() / len(original_df) * 100
        tr_p = tr_df[l].sum() / len(tr_df) * 100
        va_p = va_df[l].sum() / len(va_df) * 100
        print(f"{l:<25} {orig_p:>6.2f}%      {tr_p:>6.2f}%      {va_p:>6.2f}%")

audit_leakage_and_distributions(df, train_df, val_df)

In [ ]:
# Save split datasets
train_df.to_csv(TRAIN_OUTPUT, index=False)
val_df.to_csv(VAL_OUTPUT, index=False)
train_df.to_csv(TRAIN_TAGGED_OUTPUT, index=False)
val_df.to_csv(VAL_TAGGED_OUTPUT, index=False)

print("\n" + "="*70 + "\nSPLIT FILES SAVED\n" + "="*70)
print(f"✓ Saved primary training split:   {TRAIN_OUTPUT} ({len(train_df):,} rows)")
print(f"✓ Saved primary validation split: {VAL_OUTPUT} ({len(val_df):,} rows)")
print(f"✓ Saved tagged training split:    {TRAIN_TAGGED_OUTPUT}")
print(f"✓ Saved tagged validation split:  {VAL_TAGGED_OUTPUT}")